# Buscador de jugadores similares — Fase 1 sólida (La Liga 2018-2025)

Pipeline definitivo sobre datos **completos y estables** (temporadas cerradas 2018-2025,
congeladas en el repo archivado de worldfootballR — no cambian nunca).

**Mejoras respecto a la versión anterior:**
- Rango acotado a 2018-2025: todas las tablas completas, sin temporadas parciales.
- `passing_types` añadido al merge → features nuevas de estilo de distribución.
- Control de duplicados (traspasos de invierno) antes del merge.
- Filtro por posición compatible en ambos buscadores.
- Nombres legibles en radar y comparaciones.
- Chequeos de calidad de datos integrados.

Estructura: carga → calidad → merge → features → matrices → buscadores → visuales.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_DIR = Path("data/raw/fbref_r")
PREFIJO  = "laliga"        # cambia a "big5" cuando amplies
SEASON_MIN, SEASON_MAX = 2018, 2025   # rango cerrado y completo
MIN_MINUTOS = 900
KEYS = ["Season_End_Year", "Squad", "Comp", "Player"]

## 1. Carga con filtro de rango
Se cargan las tablas necesarias y se acota TODO a 2018-2025 desde el origen.

In [ ]:
def load(name):
    df = pd.read_csv(DATA_DIR / f"{PREFIJO}_{name}.csv")
    if "Season_End_Year" in df.columns:
        df = df[df["Season_End_Year"].between(SEASON_MIN, SEASON_MAX)]
    return df.reset_index(drop=True)

standard      = load("player_standard")
shooting      = load("player_shooting")
passing       = load("player_passing")
passing_types = load("player_passing_types")
possession    = load("player_possession")
defense       = load("player_defense")
misc          = load("player_misc")

for n, d in [("standard",standard),("shooting",shooting),("passing",passing),
             ("passing_types",passing_types),("possession",possession),
             ("defense",defense),("misc",misc)]:
    temps = sorted(d["Season_End_Year"].unique())
    print(f"{n:<15} {d.shape[0]:>6} filas | temporadas {temps[0]}-{temps[-1]} ({len(temps)})")

## 2. Chequeo de calidad: duplicados por clave
Traspasos de invierno crean 2+ filas por jugador-temporada (una por equipo). Es correcto
mantenerlas separadas (el jugador tuvo dos "medias temporadas" distintas), pero el merge
debe hacerse por la clave completa (incluye Squad) para no multiplicar filas.

In [ ]:
for n, d in [("standard",standard),("passing",passing),("possession",possession),("defense",defense)]:
    dup = d.duplicated(subset=KEYS).sum()
    print(f"{n:<15} duplicados exactos por {KEYS}: {dup}")
    if dup:
        display(d[d.duplicated(subset=KEYS, keep=False)].sort_values(KEYS).head(6))

# jugadores con 2+ equipos en una temporada (informativo, no es error)
multi = standard.groupby(["Player","Season_End_Year"])["Squad"].nunique()
print(f"\njugador-temporadas con 2+ equipos (traspasos): {(multi > 1).sum()}")

## 3. Merge de las 7 tablas
Se seleccionan columnas por tabla evitando colisiones (Int, Ast, etc. aparecen repetidas).

In [ ]:
base = standard[KEYS + ["Pos", "Nation", "Age", "Min_Playing", "Mins_Per_90_Playing",
    "Gls", "Ast", "PrgC_Progression", "PrgP_Progression", "PrgR_Progression",
    "xG_Per", "npxG_Per", "xAG_Per", "npxG+xAG_Per", "Url"]].copy()

sh = shooting[KEYS + ["Sh_per_90_Standard", "SoT_per_90_Standard", "G_per_Sh_Standard",
    "Dist_Standard", "npxG_per_Sh_Expected"]]

pa = passing[KEYS + ["Cmp_percent_Total", "KP", "PPA", "Final_Third", "xA"]]

# NUEVO: tipos de pase (estilo de distribucion)
pt = passing_types[KEYS + ["TB_Pass", "Sw_Pass", "Crs_Pass", "Live_Pass", "Dead_Pass"]]

po = possession[KEYS + ["Touches_Touches", "Att 3rd_Touches", "Att Pen_Touches",
    "Att_Take", "Succ_percent_Take", "PrgDist_Carries", "CPA_Carries",
    "Rec_Receiving", "Mis_Carries", "Dis_Carries"]]

de = defense[KEYS + ["Tkl_Tackles", "Int", "Blocks_Blocks", "Clr",
    "Tkl_percent_Challenges", "Att 3rd_Tackles"]]

mi = misc[KEYS + ["Recov", "Won_percent_Aerial", "Fld", "Fls"]]

df = (base.merge(sh, on=KEYS, how="left")
           .merge(pa, on=KEYS, how="left")
           .merge(pt, on=KEYS, how="left")
           .merge(po, on=KEYS, how="left")
           .merge(de, on=KEYS, how="left")
           .merge(mi, on=KEYS, how="left"))

assert len(df) == len(base), "el merge ha multiplicado filas: revisar duplicados"
print("tabla mergeada:", df.shape)

## 4. Features: conversión a por-90 y definición del vector
Acumulados / Mins_Per_90 → rate. Las que ya son percent/per-90 se usan directas.

In [ ]:
p90 = df["Mins_Per_90_Playing"].replace(0, np.nan)

to_per90 = ["KP", "PPA", "Final_Third", "xA",
    "TB_Pass", "Sw_Pass", "Crs_Pass",
    "Touches_Touches", "Att 3rd_Touches", "Att Pen_Touches",
    "Att_Take", "PrgDist_Carries", "CPA_Carries", "Rec_Receiving",
    "Mis_Carries", "Dis_Carries",
    "Tkl_Tackles", "Int", "Blocks_Blocks", "Clr", "Att 3rd_Tackles",
    "Recov", "Fld", "Fls",
    "PrgC_Progression", "PrgP_Progression", "PrgR_Progression"]

for c in to_per90:
    df[c + "_p90"] = pd.to_numeric(df[c], errors="coerce") / p90

already_rate = ["xG_Per", "npxG_Per", "xAG_Per", "npxG+xAG_Per",
    "Sh_per_90_Standard", "SoT_per_90_Standard", "G_per_Sh_Standard", "Dist_Standard",
    "npxG_per_Sh_Expected", "Cmp_percent_Total", "Succ_percent_Take",
    "Tkl_percent_Challenges", "Won_percent_Aerial"]

FEATURES = already_rate + [c + "_p90" for c in to_per90]
print(f"{len(FEATURES)} features de estilo")

## 5. Nombres legibles (solo para visualización)

In [ ]:
NOMBRES = {
    "xG_Per": "xG /90", "npxG_Per": "xG sin penaltis /90",
    "xAG_Per": "Asist. esperadas /90", "npxG+xAG_Per": "xG+xAG /90",
    "Sh_per_90_Standard": "Tiros /90", "SoT_per_90_Standard": "Tiros a puerta /90",
    "G_per_Sh_Standard": "Goles por tiro", "Dist_Standard": "Dist. media de tiro",
    "npxG_per_Sh_Expected": "Calidad de tiro",
    "Cmp_percent_Total": "% acierto de pase",
    "KP_p90": "Pases clave /90", "PPA_p90": "Pases al área /90",
    "Final_Third_p90": "Pases último tercio /90", "xA_p90": "xA /90",
    "TB_Pass_p90": "Pases en profundidad /90", "Sw_Pass_p90": "Cambios de juego /90",
    "Crs_Pass_p90": "Centros /90",
    "Touches_Touches_p90": "Toques /90", "Att 3rd_Touches_p90": "Toques campo rival /90",
    "Att Pen_Touches_p90": "Toques en área rival /90",
    "Att_Take_p90": "Regates intentados /90", "Succ_percent_Take": "% regates exitosos",
    "PrgDist_Carries_p90": "Dist. progresando /90", "CPA_Carries_p90": "Conducciones al área /90",
    "Rec_Receiving_p90": "Balones recibidos /90",
    "Mis_Carries_p90": "Pérdidas (control) /90", "Dis_Carries_p90": "Pérdidas (entrada) /90",
    "Tkl_Tackles_p90": "Entradas /90", "Int_p90": "Intercepciones /90",
    "Blocks_Blocks_p90": "Bloqueos /90", "Clr_p90": "Despejes /90",
    "Att 3rd_Tackles_p90": "Entradas en campo rival /90",
    "Tkl_percent_Challenges": "% duelos ganados",
    "Recov_p90": "Recuperaciones /90", "Won_percent_Aerial": "% duelos aéreos",
    "Fld_p90": "Faltas recibidas /90", "Fls_p90": "Faltas cometidas /90",
    "PrgC_Progression_p90": "Conducciones progresivas /90",
    "PrgP_Progression_p90": "Pases progresivos /90",
    "PrgR_Progression_p90": "Recepciones progresivas /90",
}
def bonito(col):
    return NOMBRES.get(col, col.replace("_", " "))

sin_nombre = [f for f in FEATURES if f not in NOMBRES]
print("features sin nombre bonito:", sin_nombre or "ninguna")

## 6. Filtros: porteros aparte, minutos mínimos, grupos de posición

In [ ]:
df["is_gk"] = df["Pos"].astype(str).str.contains("GK")

field = df[(~df["is_gk"]) & (df["Min_Playing"] >= MIN_MINUTOS)].copy()
field[FEATURES] = field[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

field["pos_group"] = field["Pos"].astype(str).str.replace(" ", "").str.split(",").str[0]
COMPATIBLES = {
    "GK": {"GK"},
    "DF": {"DF", "MF"},
    "MF": {"MF", "FW", "DF"},
    "FW": {"FW", "MF"},
}

field["label"] = field["Player"] + " (" + field["Season_End_Year"].astype(str) + ", " + field["Squad"] + ")"
field = field.reset_index(drop=True)

print(f"jugador-temporadas de campo con >= {MIN_MINUTOS} min: {len(field)}")
print(f"jugadores unicos: {field['Player'].nunique()}")
print(field["pos_group"].value_counts())

## 7. Matrices: temporada y carrera

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(field[FEATURES])
S_season = cosine_similarity(X)

Xdf = pd.DataFrame(X, columns=FEATURES)
Xdf["Player"] = field["Player"].values
career = Xdf.groupby("Player").mean()
career_labels = career.index.to_numpy()
S_career = cosine_similarity(career.values)

pos_por_jugador = field.groupby("Player")["pos_group"].agg(lambda s: s.value_counts().index[0])
career_pos = pos_por_jugador.reindex(career_labels).to_numpy()

print("matriz temporada:", S_season.shape, "| matriz carrera:", S_career.shape)

## 8. Buscador modo TEMPORADA (con filtro de posición)

In [ ]:
def similares_temporada(nombre, anyo, n=10, misma_pos=True):
    mask = (field["Player"].str.contains(nombre, case=False, na=False)) & (field["Season_End_Year"] == anyo)
    if not mask.any():
        print(f"No encuentro a '{nombre}' en {anyo} (rango valido {SEASON_MIN}-{SEASON_MAX}, min {MIN_MINUTOS} min).")
        return None
    idx = field[mask].index[0]
    sims = S_season[idx]
    orden = [i for i in np.argsort(sims)[::-1] if i != idx]
    if misma_pos:
        compat = COMPATIBLES.get(field.loc[idx, "pos_group"], {field.loc[idx, "pos_group"]})
        orden = [i for i in orden if field.loc[i, "pos_group"] in compat]
    orden = orden[:n]
    out = field.loc[orden, ["Player", "Season_End_Year", "Squad", "Pos"]].copy()
    out["similitud"] = sims[orden].round(3)
    print(f"Más parecidos a {field.loc[idx,'label']} (pos: {field.loc[idx,'pos_group']}):\n")
    return out.reset_index(drop=True)

similares_temporada("Pedri", 2025, n=10)

## 9. Buscador modo CARRERA (con filtro de posición)

In [ ]:
def similares_carrera(nombre, n=10, misma_pos=True):
    matches = [p for p in career_labels if nombre.lower() in p.lower()]
    if not matches:
        print(f"No encuentro a '{nombre}'."); return None
    player = matches[0]
    idx = list(career_labels).index(player)
    sims = S_career[idx]
    orden = [i for i in np.argsort(sims)[::-1] if i != idx]
    if misma_pos:
        compat = COMPATIBLES.get(career_pos[idx], {career_pos[idx]})
        orden = [i for i in orden if career_pos[i] in compat]
    orden = orden[:n]
    out = pd.DataFrame({"Player": career_labels[orden],
                        "pos": career_pos[orden],
                        "similitud": sims[orden].round(3)})
    print(f"Perfil de carrera más parecido a {player} (pos: {career_pos[idx]}):\n")
    return out

similares_carrera("Pedri", n=10)

## 10. Radar de comparación (percentiles, nombres legibles)

In [ ]:
def radar(nombre_a, anyo_a, nombre_b, anyo_b, features=None):
    def fila(nombre, anyo):
        m = (field["Player"].str.contains(nombre, case=False, na=False)) & (field["Season_End_Year"] == anyo)
        return field[m].index[0] if m.any() else None
    ia, ib = fila(nombre_a, anyo_a), fila(nombre_b, anyo_b)
    if ia is None or ib is None:
        print("No encuentro a uno de los dos."); return None

    if features is None:
        features = ["xG_Per", "xAG_Per", "Sh_per_90_Standard", "Cmp_percent_Total",
                    "KP_p90", "PPA_p90", "TB_Pass_p90", "PrgP_Progression_p90",
                    "PrgC_Progression_p90", "Succ_percent_Take",
                    "Tkl_Tackles_p90", "Int_p90"]
    features = [f for f in features if f in FEATURES]

    pct = field[features].rank(pct=True)
    va = np.concatenate([pct.loc[ia].values * 100, [pct.loc[ia].values[0] * 100]])
    vb = np.concatenate([pct.loc[ib].values * 100, [pct.loc[ib].values[0] * 100]])
    ang = np.linspace(0, 2*np.pi, len(features), endpoint=False).tolist()
    ang += ang[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.plot(ang, va, linewidth=2, label=field.loc[ia, "label"], color="crimson")
    ax.fill(ang, va, alpha=0.15, color="crimson")
    ax.plot(ang, vb, linewidth=2, label=field.loc[ib, "label"], color="steelblue")
    ax.fill(ang, vb, alpha=0.15, color="steelblue")
    ax.set_xticks(ang[:-1]); ax.set_xticklabels([bonito(f) for f in features], fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_title(f"Percentiles vs jugadores de campo La Liga {SEASON_MIN}-{SEASON_MAX}", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
    plt.tight_layout(); plt.show()

radar("Pedri", 2025, "Frenkie de Jong", 2025)

## 11. Radar solo ofensivo (para comparar atacantes)

In [ ]:
RADAR_OFENSIVO = ["xG_Per", "npxG_Per", "Sh_per_90_Standard", "SoT_per_90_Standard",
    "npxG_per_Sh_Expected", "xAG_Per", "KP_p90", "PPA_p90",
    "Att Pen_Touches_p90", "Att_Take_p90", "Succ_percent_Take", "CPA_Carries_p90"]

# ejemplo: dos delanteros
radar("Lewandowski", 2025, "Dovbyk", 2024, features=RADAR_OFENSIVO)

## 12. Mapa 2D con dos jugadores resaltados
Recordatorio: la distancia visual NO es la similitud real (el PCA comprime 40D→2D).
El número fiable es el coseno del título.

In [ ]:
def mapa_dos_jugadores(nombre_a, anyo_a, nombre_b, anyo_b):
    def fila(nombre, anyo):
        m = (field["Player"].str.contains(nombre, case=False, na=False)) & (field["Season_End_Year"] == anyo)
        return field[m].index[0] if m.any() else None
    ia, ib = fila(nombre_a, anyo_a), fila(nombre_b, anyo_b)
    if ia is None or ib is None:
        print("No encuentro a uno de los dos."); return None

    coords = PCA(n_components=2, random_state=0).fit_transform(X)
    fig, ax = plt.subplots(figsize=(11, 8))
    for pos, g in field.groupby("pos_group"):
        sel = g.index
        ax.scatter(coords[sel, 0], coords[sel, 1], s=8, alpha=0.15, label=pos)
    for idx, color, dy in [(ia, "crimson", 14), (ib, "steelblue", -22)]:
        ax.scatter(coords[idx, 0], coords[idx, 1], s=220, color=color, edgecolor="black", zorder=5)
        ax.annotate(field.loc[idx, "label"], (coords[idx, 0], coords[idx, 1]),
                    fontsize=10, fontweight="bold", xytext=(8, dy), textcoords="offset points",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=color, alpha=0.9))
    sim = cosine_similarity(X[ia].reshape(1,-1), X[ib].reshape(1,-1))[0,0]
    ax.set_xlim(np.percentile(coords[:,0], 1), np.percentile(coords[:,0], 99))
    ax.set_ylim(np.percentile(coords[:,1], 1), np.percentile(coords[:,1], 99))
    ax.set_title(f"Similitud coseno = {sim:.3f}  |  PCA 2D de {len(FEATURES)} features")
    ax.legend(title="posición", markerscale=2, fontsize=8)
    plt.tight_layout(); plt.show()

mapa_dos_jugadores("Pedri", 2025, "Frenkie de Jong", 2025)

## Pendiente / siguientes pasos
- **Big 5**: re-ejecuta el notebook de R con `SOLO_LALIGA <- FALSE` y cambia `PREFIJO = "big5"` aquí.
- **Porteros**: mismo pipeline con `keepers`/`keepers_adv` y features de portero (PSxG, % paradas, juego de pies).
- **UMAP** para el mapa: mejor preservación de vecindad que PCA.
- **PostgreSQL**: volcar `field[KEYS + FEATURES]` a `core` como base de `player_ratings`.
- **Temporada en curso (fase 2)**: fuente pendiente de decidir (repo archivado el 18/09/2025).